In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 四配置公开视频 Baseline v4

选择 GPU 后 Run all。固定 UCF101 DEV64 / CAL500 / TEST500，181 连续真实帧、25fps、320×512；40 视图。WAM32、PixelSeal256、VideoSeal v1.0 256 和 ADAPTED PixelSeal-FrameIndex256（248 owner＋8 index）分行报告。原生三个方法的时间列为 N/A。

0.1% 是逐方法、单随机 query、固定40视图族的名义上界，不是 TEST500 对自然总体的经验认证；0/500 单侧95%上界约0.597%。这批不生成 Wan，也不代表主方法公平排名已经完成。


In [ ]:
SOURCE_REF = 'dev/baselines-paper-v4'
SOURCE_DIRECTORY = None  # 可选：可编辑的本地源码目录
PREVIOUS_RUNTIME_STOPPED = False  # 仅旧的不同主机 runtime 已实际停止时设 True


## 执行与续接

同一 Drive 结果包重开 Run all，从缺失/中断部分继续。此前保存结果默认完整。直接读取 compute_checkpoint.json 续接；旧包首次仅用 runner.log 的真实计算位置和当前任务调用记录迁移，不扫描历史源/40视图状态，不重建完成索引。恢复扫描进度不作为计算断点。已提交批次直接复用，实际需要的数据读取失败时报告具体错误。每次从分支下载新的临时源码，避免旧 project 副本遮蔽更新。启动不再扫描历史临时目录；本次自有源码在worker停止后释放，正式结果与恢复记录不动。先保存原生读数，再运行 query 验证及 CPU 后评。每实际批次数组即时保存；已有完整批次不重复推理。若同一 view 的接收像素缓存丢失，旧 partial 保留，该 view 在固定恢复预算内整份重建，不混合两份输入。每物理单位最多2次额外基础设施重算、各类累计最多基础上限2%；方法失败不换参数重跑。最多64次新 runtime 续接。

同一包只运行一个实例。正常关闭后无需修改设置；若旧不同主机仍标记活动，请先实际停止它再确认续接。心跳不是跨主机锁，不按超时抢写。监视器只把本次 worker 的进度标为当前；旧行明确显示为历史，无变化会显示停留时长。旧任务仍在运行时先停止并断开旧运行时；新运行时续接时将 PREVIOUS_RUNTIME_STOPPED 设为 True，再 Run all。

正式及失败结果位于 MyDrive/Video-WM_Result/baselines/public-native-and-frameindex-fpr001-v4/attempt01。源码、权重和衍生缓存位于其外。已有 Drive 空间未知，新增空间约200–260GB只是规划；DEV供体解码缓存约28.5GB、实际媒体大小另计，空间不足暂停未开始单元，保留正式包。没有预先测速，不承诺GPU小时或单次Colab完成。


In [ ]:
from pathlib import Path
import urllib.request
bootstrap_path = 'runtime/wan/native_baseline_colab_v4.py'
if SOURCE_DIRECTORY:
    bootstrap_text = (Path(SOURCE_DIRECTORY) / bootstrap_path).read_text()
else:
    url = f'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/{SOURCE_REF}/{bootstrap_path}'
    bootstrap_text = urllib.request.urlopen(url).read().decode('utf-8')
entry = {'__name__': 'native_baseline_v4_colab_entry'}
exec(compile(bootstrap_text, bootstrap_path, 'exec'), entry)
RUN = entry['run_colab'](source_directory=SOURCE_DIRECTORY, source_ref=SOURCE_REF, previous_runtime_stopped=PREVIOUS_RUNTIME_STOPPED)


## 读取四类结果

A：原生 payload BER、完整消息、coverage 与擦除；B：给定query验证、OFF FP/区间、wrong-query及固定DEV跨源读量复用；C：FrameIndex位置/邻边/错误归属与未决；D：precodec、G1、G2-identity画质和实际资源。全计划分母及失败不删除。partial WAM/FrameIndex阳性见证保留为TRUE但未完整；Seal缺一计划帧时视频均值验证为UNKNOWN。

此 notebook 交付时未启动真实模型、codec、GPU或Colab实验。工程检查不等于科学效果。


In [ ]:
print('入口状态:', RUN['status'])
print('保留结果:', RUN['output'])
entry['show_results']()
